In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_MAP = {
    'normal'                        : 'Normal',
    'anomalous(DoSattack)'          : 'DoS',
    'anomalous(scan)'               : 'Scan',
    'anomalous(malitiousControl)'   : 'MaliciousControl',
    'anomalous(malitiousOperation)' : 'MaliciousOperation',
    'anomalous(spying)'             : 'Spying',
    'anomalous(dataProbing)'        : 'DataProbing',
    'anomalous(wrongSetUp)'         : 'WrongSetUp',
}
CLASS_TO_INT = {v: i for i, v in enumerate(CLASS_MAP.values())}
INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}

SEARCH_PATHS = [
    Path("DS2OS.csv"),
    Path.home() / "Downloads" / "DS2OS.csv",
    (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv"),
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError("DS2OS.csv no encontrado.")

df_raw = pd.read_csv(CSV_PATH)
print(f"Shape: {df_raw.shape}")

Shape: (357952, 13)


In [2]:
df = df_raw.copy()

df['accessedNodeType'] = df['accessedNodeType'].fillna('Malicious')
df['value'] = df['value'].replace({'False': 0, 'True': 1, 'Twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
df = df.drop(columns=['timestamp'])
df['y'] = df['normality'].map(CLASS_MAP).map(CLASS_TO_INT).astype(np.int8)
df = df.drop(columns=['normality'])

CAT_COLS = [
    'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
    'destinationServiceAddress', 'destinationServiceType',
    'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation',
]
for col in CAT_COLS:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEATURE_COLS = CAT_COLS + ['value']
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['y'].values

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_tr, X_val, y_tr, y_val   = train_test_split(X_tv, y_tv, test_size=0.20, random_state=SEED, stratify=y_tv)

scaler = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_te_scaled  = scaler.transform(X_test)

N_FEATURES = X_tr_scaled.shape[1]
print(f"Train: {X_tr_scaled.shape} | Val: {X_val_scaled.shape} | Test: {X_te_scaled.shape}")

Train: (229088, 11) | Val: (57273, 11) | Test: (71591, 11)


In [3]:
def build_autoencoder(input_dim, latent_dim=3):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='tanh')(inputs)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(6,  activation='tanh')(x)
    latent  = layers.Dense(latent_dim, activation='tanh', name='latent')(x)
    x = layers.Dense(6,  activation='tanh')(latent)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(32, activation='tanh')(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inputs, outputs, name='AE_Van2020')
    encoder     = Model(inputs, latent,  name='Encoder_Van2020')
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(input_dim=N_FEATURES, latent_dim=3)
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
autoencoder.fit(
    X_tr_scaled, X_tr_scaled,
    validation_data=(X_val_scaled, X_val_scaled),
    epochs=50, batch_size=256,
    callbacks=[early_stop], verbose=1
)

Z_train = encoder.predict(X_tr_scaled,  verbose=0)
Z_val   = encoder.predict(X_val_scaled, verbose=0)
Z_test  = encoder.predict(X_te_scaled,  verbose=0)
print(f"Espacio latente — Train: {Z_train.shape} | Val: {Z_val.shape} | Test: {Z_test.shape}")

dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid={'max_depth': [3, 5, 10, 15, None], 'min_samples_split': [2, 5, 10], 'criterion': ['gini', 'entropy']},
    cv=5, scoring='f1_weighted', n_jobs=-1
)
dt_gs.fit(Z_train, y_tr)
best_dt = dt_gs.best_estimator_
print(f"DT  — mejores params: {dt_gs.best_params_} | CV F1: {dt_gs.best_score_:.4f}")

knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={'n_neighbors': [3, 5, 7, 10, 15], 'metric': ['euclidean', 'manhattan'], 'weights': ['uniform', 'distance']},
    cv=5, scoring='f1_weighted', n_jobs=-1
)
knn_gs.fit(Z_train, y_tr)
best_knn = knn_gs.best_estimator_
print(f"kNN — mejores params: {knn_gs.best_params_} | CV F1: {knn_gs.best_score_:.4f}")

best_f1, best_svm, best_C = -1, None, None
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = CalibratedClassifierCV(LinearSVC(C=C, max_iter=2000, random_state=SEED), cv=3)
    clf.fit(Z_train, y_tr)
    f1 = f1_score(y_val, clf.predict(Z_val), average='weighted')
    if f1 > best_f1:
        best_f1, best_svm, best_C = f1, clf, C
print(f"SVM — mejor C: {best_C} | F1 val: {best_f1*100:.2f}%")

classifiers = {'DT': best_dt, 'kNN': best_knn, 'SVM': best_svm}

Epoch 1/50
895/895 [==============================] - 8s 8ms/step - loss: 0.2543 - val_loss: 0.0982
Epoch 2/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0612 - val_loss: 0.0270
Epoch 3/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0208 - val_loss: 0.0159
Epoch 4/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0156 - val_loss: 0.0139
Epoch 5/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0139 - val_loss: 0.0126
Epoch 6/50
895/895 [==============================] - 4s 5ms/step - loss: 0.0119 - val_loss: 0.0096
Epoch 7/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0091 - val_loss: 0.0078
Epoch 8/50
895/895 [==============================] - 5s 6ms/step - loss: 0.0077 - val_loss: 0.0071
Epoch 9/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0067 - val_loss: 0.0058
Epoch 10/50
895/895 [==============================] - 3s 3ms/step - loss: 0.0059 - val_loss: 0.0051

In [4]:
SEP = '─' * 58
print("Autoencoder + DT/kNN/SVM — Dataset DS2OS")
print(SEP)
print(f"{'Modelo':<6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
print(f"{'─'*6} {'─'*6} {'─'*6} {'─'*6}")

for name, clf in classifiers.items():
    y_pred = clf.predict(Z_test)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score       (y_test, y_pred, average='weighted', zero_division=0)
    print(f"{name:<6} {prec:.3f}  {rec:.3f}  {f1:.3f}")

print(SEP)
target_names = list(INT_TO_CLASS.values())
print(classification_report(y_test, best_dt.predict(Z_test), target_names=target_names, zero_division=0))

Autoencoder + DT/kNN/SVM — Dataset DS2OS
──────────────────────────────────────────────────────────
Modelo   Prec    Rec     F1
────── ────── ────── ──────
DT     0.995  0.995  0.994
kNN    0.994  0.994  0.994
SVM    0.945  0.972  0.958
──────────────────────────────────────────────────────────
                    precision    recall  f1-score   support

            Normal       0.99      1.00      1.00     69588
               DoS       0.98      0.68      0.80      1156
              Scan       1.00      1.00      1.00       310
  MaliciousControl       1.00      1.00      1.00       178
MaliciousOperation       0.99      1.00      1.00       161
            Spying       0.99      1.00      1.00       106
       DataProbing       1.00      1.00      1.00        68
        WrongSetUp       1.00      1.00      1.00        24

          accuracy                           0.99     71591
         macro avg       1.00      0.96      0.97     71591
      weighted avg       0.99      0.99   